# Water Quality Prediction: Benchmark Notebook 

## Challenge Overview

Welcome to the EY AI & Data Challenge 2026!  
The objective of this challenge is to build a robust **machine learning model** capable of predicting water quality across various river locations in South Africa. In addition to accurate predictions, the model should also identify and emphasize the key factors that significantly influence water quality.

Participants will be provided with a dataset containing three water quality parameters — **Total Alkalinity**, **Electrical Conductance**, and **Dissolved Reactive Phosphorus** — collected between 2011 and 2015 from approximately 200 river locations across South Africa. Each data point includes the geographic coordinates (latitude and longitude) of the sampling site, the date of collection, and the corresponding water quality measurements.

Using this dataset, participants are expected to build a machine learning model to predict water quality parameters for a separate validation dataset, which includes locations from different regions not present in the training data. The challenge also encourages participants to explore feature importance and provide insights into the factors most strongly associated with variations in water quality.

This challenge is designed for participants with varying levels of experience in data science, remote sensing, and environmental analytics. It offers a valuable opportunity to apply machine learning techniques to real-world environmental data and contribute to advancing water quality monitoring using artificial intelligence.


**About the Notebook:**  

In this notebook, we demonstrate an advanced workflow that serves as a foundation for the challenge. The model predicts **water quality parameters** using features derived from the **Landsat** and **TerraClimate** datasets. 

Specifically, four spectral bands — **SWIR22** (Shortwave Infrared 2), **NIR** (Near Infrared), **Green**, and **SWIR16** (Shortwave Infrared 1) — were utilized from Landsat, along with derived spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). In addition, the **PET** (Potential Evapotranspiration) variable was incorporated from the **TerraClimate** dataset to account for climatic influences on water quality.

The dataset spans a five-year period from **2011 to 2015**. Using **API-based data extraction** methods, both Landsat and TerraClimate features were retrieved directly from the [Microsoft Planetary Computer portal](https://planetarycomputer.microsoft.com).

These combined spectral, index-based, and climatic features were used as predictors in a regression model to estimate three key water quality parameters: **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.

For Phase 2, we have expanded the climatic features from TerraClimate to include PET (Potential Evapotranspiration), PR (Precipitation), and TMAX (Maximum Temperature). These variables provide a more comprehensive view of the hydrological and thermal conditions affecting water quality.


## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 

## Response Variable

Before building the model, we first load the **water quality training dataset**. The curated dataset contains samples collected from various monitoring stations across the study region. Each record includes the geographical coordinates (Latitude and Longitude), the sample collection date, and the corresponding **measured values** for the three key water quality parameters — **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.


In [ ]:
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
display(Water_Quality_df.head(5))

### Overview del dataset

In [ ]:
print("Shape:", Water_Quality_df.shape)
Water_Quality_df.info()
Water_Quality_df.describe()

### Missing values

In [ ]:
Water_Quality_df.isna().sum().sort_values(ascending=False)

### Target distribution

In [ ]:
import matplotlib.pyplot as plt

targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

for col in targets:
    plt.figure()
    Water_Quality_df[col].hist(bins=50)
    plt.title(col)
    plt.show()

### Outliers

In [ ]:
for col in targets:
    print(col)
    print(Water_Quality_df[col].quantile([0.01,0.99]))
    print()

### Temporal analysis

In [ ]:
Water_Quality_df["Sample Date"] = pd.to_datetime(
    Water_Quality_df["Sample Date"],
    dayfirst=True  # importante: interpreta dd-mm-yyyy
)

# Luego extraemos año y mes
Water_Quality_df["year"] = Water_Quality_df["Sample Date"].dt.year
Water_Quality_df["month"] = Water_Quality_df["Sample Date"].dt.month

In [ ]:
Water_Quality_df.groupby("month")[targets].mean().plot()
plt.title("Seasonality")
plt.show()

### Spatial sanity check

In [ ]:
plt.figure()
plt.scatter(
    Water_Quality_df["Longitude"],
    Water_Quality_df["Latitude"],
    alpha=0.3
)
plt.title("Sampling locations")
plt.show()

### Correlation between targets

In [ ]:
import seaborn as sns

sns.heatmap(
    Water_Quality_df[targets].corr(),
    annot=True,
    cmap="coolwarm"
)
plt.title("Target correlations")
plt.show()

In [ ]:
Water_Quality_df.groupby(
    Water_Quality_df["Latitude"].round(1)
)[targets].mean()

## Predictor Variables

Now that we have our water quality dataset, the next step is to gather the predictor variables from the **Landsat** and **TerraClimate** datasets. In this notebook, we demonstrate how to **load previously extracted satellite and climate data** from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. We can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.

For more detailed guidance on the original data extraction process, you can review the Landsat and TerraClimate example notebooks available on the Planetary Computer portal:

- [Landsat-c2-l2 - Example-Notebook](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook)  
- [Terraclimate - Example-Notebook](https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook)

We have used selected spectral bands — **SWIR22** (Shortwave Infrared 2), **NIR** (Near Infrared), **Green**, and **SWIR16** (Shortwave Infrared 1) — and computed key spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, and water content characteristics that influence water quality variability.

In addition to Landsat features, we also incorporated the **Potential Evapotranspiration (PET)** variable from the **TerraClimate** dataset, which provides high-resolution global climate data. The PET feature captures the atmospheric demand for moisture, representing climatic conditions such as temperature, humidity, and radiation that influence surface water evaporation and thus affect water quality parameters.

The predictor features include:

- **SWIR22** – Sensitive to surface moisture and turbidity variations in water bodies.  
- **NIR** – Helps in identifying vegetation and suspended matter in water.  
- **Green** – Useful for detecting water color and surface reflectance changes.  
- **SWIR16** – Provides information on surface dryness and sediment concentration.  
- **NDMI** – Derived from NIR and SWIR16, indicates moisture and vegetation–water interaction.  
- **MNDWI** – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.  
- **PET** – Extracted from the TerraClimate dataset, represents potential evapotranspiration influencing hydrological and water quality dynamics.
- **PR** (Precipitation) – Extracted from TerraClimate, this variable accounts for rainfall events that lead to runoff, dilution, or increased sediment transport into water bodies.
- **TMAX** (Maximum Temperature) – Represents the maximum monthly temperature, influencing water temperature, biological activity, and evaporation rates.


### Loading Pre-Extracted Landsat Data

This notebook adopts a modular pipeline architecture by ingesting pre-extracted Landsat features from CSV files generated in the dedicated extraction stage. By decoupling data acquisition from the modeling workflow, we bypass the significant latency associated with large-scale satellite API queries, enabling rapid iteration and model experimentation.

Participants should first execute the companion Landsat Data Extraction Notebook to generate the required feature sets. This process calculates vital spectral bands and specialized indices—such as NDMI, MNDWI, and Turbidity proxies—specifically optimized for water quality assessment. Leveraging these pre-processed datasets follows best practices for managing high-volume environmental data in machine learning projects


#### Note

In the data extraction process (performed in the dedicated extraction notebooks), a 100 m focal buffer was applied around each sampling location rather than using a single point. Participants may explore creating different focal buffers around the locations (e.g., 50 m, 150 m, etc.) during extraction. For example, if a 50 m buffer was used for “Band 2”, the extracted CSV values would reflect the average of Band 2 within 50 meters of each location. This approach can help reduce errors associated with spatial autocorrelation.


In [ ]:
landsat_train_features = pd.read_csv("landsat_features_training_fase2.csv")
display(landsat_train_features.head(5))

In [ ]:
# If NDMI and MNDWI columns are of type object, convert them to float
landsat_train_features['NDMI'] = landsat_train_features['NDMI'].astype(float)
landsat_train_features['MNDWI'] = landsat_train_features['MNDWI'].astype(float)


In [ ]:
print("Variables Landsat:", landsat_train_features.columns.tolist())

### Loading Pre-Extracted TerraClimate Data

This notebook integrates high-resolution climatic data by ingesting pre-processed TerraClimate features from local datasets. By utilizing a decoupled extraction workflow, we eliminate the computational overhead of large-scale Zarr dataset processing, facilitating rapid model iteration and experimentation.

Participants should first execute the companion TerraClimate Data Extraction Notebook to derive key hydrological drivers, including Potential Evapotranspiration (PET), Precipitation (PR), and Maximum Temperature (TMAX). This modular approach ensures consistent, high-fidelity climate inputs that are essential for multi-modal environmental analysis and robust water quality modeling


In [ ]:
Terraclimate_df = pd.read_csv("terraclimate_features_training_fase2.csv")
display(Terraclimate_df.head(5))

In [ ]:
print("Variables TerraClimate:", Terraclimate_df.columns.tolist())

## Joining the Predictor Variables and Response Variables

Now that we have extracted our predictor variables, we need to join them with the response variables. We use the **combine_two_datasets** function to merge the predictor variables and response variables. The **concat** function from pandas is particularly useful for this step.


In [5]:
# Combine two datasets vertically (along columns) using pandas concat function.
def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [6]:
# Combining ground data and final data into a single dataset.
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
display(wq_data.head(5))

### Handling Missing Values

Before model training, missing values in the dataset were carefully handled to ensure data consistency and prevent model bias. Numerical columns were imputed using their median values, maintaining the overall data distribution while minimizing the impact of outliers.


In [7]:
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
wq_data.isna().sum()

## Model Building

Now let us select the columns required for our model-building exercise. We will consider only **SWIR22**, **NDMI**, and **MNDWI** from the Landsat data, and **PET** from the TerraClimate dataset as our predictor variables. It does not make sense to use latitude and longitude as predictor variables, as they do not have any direct impact on predicting the water quality parameters.


In [ ]:
# We selected the Landsat variables and the THREE from TerraClimate
# Note: Make sure the names match your CSV (pet, pr, tmax)
features_clima = ['pet', 'pr', 'tmax']
features_landsat = ['swir22', 'NDMI', 'MNDWI']
targets = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# We filtered the combined dataset
wq_data = wq_data[features_landsat + features_clima + targets]

display(wq_data.head())

In [ ]:
# Uploading validation files Phase 2
landsat_val_features = pd.read_csv("landsat_features_validation_fase2.csv")
Terraclimate_val_df = pd.read_csv("terraclimate_features_validation_fase2.csv")

# Consolidate the validation dataframe with the new variables
val_data = pd.DataFrame({
    'Longitude': landsat_val_features['Longitude'].values,
    'Latitude': landsat_val_features['Latitude'].values,
    'Sample Date': landsat_val_features['Sample Date'].values,
    'swir22': landsat_val_features['swir22'].values,
    'NDMI': landsat_val_features['NDMI'].values,
    'MNDWI': landsat_val_features['MNDWI'].values,
    'pet': Terraclimate_val_df['pet'].values,
    'pr': Terraclimate_val_df['pr'].values,   # Nueva
    'tmax': Terraclimate_val_df['tmax'].values # Nueva
})

# Impute any null values.
val_data = val_data.fillna(val_data.median(numeric_only=True))

# Select only the training columns
submission_val_data = val_data[features_landsat + features_clima]
display(submission_val_data.head())

### **Notes**

We implement a multi-target modeling strategy, developing specialized regressors for each water quality parameter. The core feature space integrates high-resolution satellite-derived indices (SWIR22, NDMI, MNDWI) with climatic drivers (PET, PR, and TMAX). This multi-modal approach is designed to capture the complex interactions between land-surface reflectance and hydrological conditions. While this notebook establishes a baseline configuration, participants are encouraged to iterate on feature selection and explore advanced architectures to enhance model generalizability


## Helper Functions and Machine Learning Pipeline

### Train and Test Split
To evaluate the model's ability to generalize to unseen data, we partition the dataset into **70% training** and **30% testing** sets. This is achieved using the `train_test_split` utility from the `scikit-learn` `model_selection` module. This split ensures that we have a distinct hold-out set to validate the model's performance before final submission.

### Feature Scaling (Standardization)
Numerical feature scaling is a vital preprocessing step to ensure that all predictors contribute equally to the model's decision-making process. We utilize the **StandardScaler** from `scikit-learn`, which transforms each feature—including **SWIR22, NDMI, MNDWI, PET, PR, and TMAX**—to have a mean of 0 and a standard deviation of 1.

Standardization is particularly important for gradient-based and boosting algorithms, as it stabilizes the learning process and improves convergence speed.

### Multi-Target Model Training
Our approach involves building three independent regression models, one for each target water quality parameter: **Total Alkalinity**, **Electrical Conductance**, and **Dissolved Reactive Phosphorus**. 

For **Phase 2**, we have upgraded our training strategy. While we maintain a **Random Forest Regressor** as a performance baseline, we have introduced the **HistGradientBoostingRegressor**. This modern algorithm incorporates native L2 regularization, making it significantly more robust against the high-variance nature of satellite and climatic data.

Predictor variables (X) are strictly limited to spectral and climatic features, while metadata such as Latitude, Longitude, and Sample Date are excluded to prevent the model from "leaking" spatio-temporal references instead of learning biophysical relationships.

### Model Evaluation and Generalization Analysis
Performance is quantified using two primary metrics:
- **R² Score**: Measures how well the model explains the variance in the observed values.  
- **RMSE (Root Mean Square Error)**: Quantifies the average magnitude of prediction errors.

Furthermore, we perform a **Generalization Gap Analysis**. By comparing the $R^2$ scores of the training set against the test set, we can identify and mitigate **overfitting**. A model with a smaller gap between these scores is considered more stable and reliable for the final challenge submission.




In [ ]:
def get_trained_model(X_train, y_train, version="overfit"):
    if version == "overfit":
        # The original model that memorized everything
        model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    else:
        # The regularized model (honest)
        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=5,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1
        )
    
    model.fit(X_train, y_train)
    return model

In [9]:
def split_data(X, y, test_size=0.3, random_state=42):
    """"Divide the data into training and test."""
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    """Scale the variables to normalize the input ranges."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train, y_train, version="HistGradientBoosting"):
    """
Train the model using the chosen algorithm:
- 'RandomForest': RandomForestRegressor (Base configuration).
- 'HistGradientBoosting': HistGradientBoostingRegressor (Robust configuration).
    """
    if version == "RandomForest":
        # Model based on assembly of independent trees
        model = RandomForestRegressor(
            n_estimators=100, 
            random_state=42, 
            n_jobs=-1
        )
    elif version == "HistGradientBoosting":
        # Model based on boosting (gradient) with regularization
        model = HistGradientBoostingRegressor(
            max_iter=300,
            max_depth=5,           
            learning_rate=0.05,
            l2_regularization=1.5, 
            random_state=42
        )
    else:
        print(f"⚠️ Version '{version}' not recognized. Using HistGradientBoosting.")
        return train_model(X_train, y_train, version="HistGradientBoosting")
        
    model.fit(X_train, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    """Evaluate the model and display error metrics."""
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Clean print format for comparison
    print(f"| {dataset_name:^12} | R²: {r2:7.3f} | RMSE: {rmse:9.3f} |")
    return y_pred, r2, rmse

## Model Workflow (Pipeline)

The complete model development process follows a structured pipeline to ensure consistency, reproducibility, and clarity. Each stage in the workflow is modularized into independent functions that can be reused for different water quality parameters. This modular approach streamlines the process and makes the workflow easily adaptable to new datasets or parameters in the future.

The pipeline automates the sequence of steps — from data preparation to evaluation — for each target parameter. The same set of predictor variables is used, while the response variable changes for each of the three targets: *Total Alkalinity (TA)*, *Electrical Conductance (EC)*, and *Dissolved Reactive Phosphorus (DRP)*. By maintaining a consistent framework, comparisons across models remain fair and interpretable.


In [10]:
def run_comparison_pipeline(X, y, param_name, model_version):
    """
Run the complete ML flow for a specific version of the model
and return the results for comparison.
    """
    print(f"\n>>> Procesando {param_name} | Versión: {model_version.upper()} <<<")
    
    # 1. Data split
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # 2. Data scaling
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    
    # 3. Model training (using the version parameter: 'overfit' or 'stable')
    model = train_model(X_train_scaled, y_train, version=model_version)
    
    # 4. Evaluation and metric capture
    # The evaluate_model function already prints the results to the console
    _, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, f"Train ({model_version})")
    _, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, f"Test ({model_version})")
    
    # 5. Return dictionary to build the comparative DataFrame
    return {
        "Parameter": param_name,
        "Version": model_version,
        "R2_Train": r2_train,
        "R2_Test": r2_test,
        "RMSE_Train": rmse_train,
        "RMSE_Test": rmse_test
    }, model, scaler

### Model Training and Evaluation for Each Parameter

In this step, we apply the complete modeling pipeline to each of the three selected water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. The input feature set (`X`) remains the same across all three models, while the target variable (`y`) changes for each parameter. 

For every parameter, the `run_pipeline()` function is executed, which handles data preprocessing, model training, and both in-sample and out-of-sample evaluation. This ensures a consistent workflow and allows for a fair comparison of model performance across different water quality indicators.


In [ ]:
# 1. Define the objectives and prepare the data
targets = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
versiones = ['RandomForest', 'HistGradientBoosting']
X = wq_data.drop(columns=targets)

# Lists for saving results and dictionaries for the final models
all_comparison_results = []
final_models = {} 

# 2. Comparative training loop
for target_name in targets:
    y = wq_data[target_name]
    X = wq_data.drop(columns=targets)
    
    for v in versiones:
        # Run the pipeline using the new names
        res, model, scaler = run_comparison_pipeline(X, y, target_name, v)
        all_comparison_results.append(res)
        
        # Save the HistGradientBoosting as our production model
        if v == 'HistGradientBoosting':
            final_models[target_name] = {"model": model, "scaler": scaler}

# 3. Show the final comparison table
df_comparison = pd.DataFrame(all_comparison_results)

# Direct contrast between versions for each parameter.
print("\n" + "="*60)
print("COMPARATIVE SUMMARY OF MODELS")
print("="*60)
display(df_comparison.sort_values(by=["Parameter", "Version"]))

### Model Performance Summary

After training and evaluating the models for each water quality parameter, the individual performance metrics are combined into a single summary table. This table consolidates the R² and RMSE values for both in-sample and out-of-sample evaluations, enabling an easy comparison of model performance across Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. 

Such a summary provides a quick overview of how well each model captures the variability in each parameter and highlights any differences in predictive accuracy.


In [12]:
# The 'df comparison' DataFrame already contains the results for all models and versions
results_summary = df_comparison.sort_values(by=["Parameter", "Version"])
results_summary

## Submission

Once you are satisfied with your model’s performance, you can proceed to make predictions for unseen data. To do this, use your trained model to estimate the concentrations of the target water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus — for a set of test locations provided in the **Submission_template.csv** file. 

The predicted results can then be uploaded to the challenge platform for evaluation.


In [ ]:
test_file = pd.read_csv("submission_template.csv")
display(test_file.head(5))

In [ ]:
landsat_val_features = pd.read_csv("landsat_features_validation.csv")
display(landsat_val_features.head(5))

In [ ]:
Terraclimate_val_df = pd.read_csv("terraclimate_features_validation_fase2.csv")
display(Terraclimate_val_df.head(5))

#### Notes

Similarly, the Landsat and TerraClimate extraction modules should be used to generate feature sets for the validation data. To facilitate immediate testing, we provide pre-computed sample outputs (e.g., landsat_features_validation_fase2.csv and terraclimate_features_validation_fase2.csv).

For the inference pipeline to function correctly, custom-extracted files must adhere to the established naming convention and column schema. Maintaining this consistency allows the benchmark notebook to automatically ingest validation features and execute the prediction stage without manual intervention.


In [16]:
# Consolidate all the extracted bands and features in a single dataframe
val_data = pd.DataFrame({
    'Longitude': landsat_val_features['Longitude'].values,
    'Latitude': landsat_val_features['Latitude'].values,
    'Sample Date': landsat_val_features['Sample Date'].values,
    'swir22': landsat_val_features['swir22'].values,
    'NDMI': landsat_val_features['NDMI'].values,
    'MNDWI': landsat_val_features['MNDWI'].values,
    'pet': Terraclimate_val_df['pet'].values,
    'pr': Terraclimate_val_df['pr'].values,   # Nueva
    'tmax': Terraclimate_val_df['tmax'].values # Nueva
})

In [17]:
# Impute the missing values
val_data = val_data.fillna(val_data.median(numeric_only=True))

In [18]:
# Extracting specific columns (B01, B06, and NDVI) from the validation dataset
features_model = ['swir22', 'NDMI', 'MNDWI', 'pet', 'pr', 'tmax']
submission_val_data = val_data[features_model]
display(submission_val_data.head())
print(f"Formato final para predicción: {submission_val_data.shape}")

In [20]:
# Dictionary for storing weather predictions
preds_dict = {}
targets = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

for target in targets:
    # We extract the 'stable' model and scaler from the dictionary.
    model = final_models[target]['model']
    scaler = final_models[target]['scaler']
    
    # Scale and predict
    X_val_scaled = scaler.transform(submission_val_data)
    y_pred = model.predict(X_val_scaled)
    
    # Store predictions ensuring no negative values
    preds_dict[target] = np.maximum(0, y_pred)

print("✅ Predictions calculated for all parameters.")

In [21]:
# We identify the ID of the validation column (usually dis_detail_id or ID)
col_id = next((c for c in ['dis_detail_id', 'ID', 'id'] if c in test_file.columns), test_file.columns[0])

# We created the submission DataFrame
submission_df = pd.DataFrame({
    col_id: test_file[col_id].values,
    'Total Alkalinity': preds_dict['Total Alkalinity'],
    'Electrical Conductance': preds_dict['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': preds_dict['Dissolved Reactive Phosphorus']
})

# Local save and climb to Snowflake
submission_df.to_csv("/tmp/submission.csv", index=False)


In [ ]:
# If you use Snowflake (PUT), hold this command
session.sql("""
    PUT file:///tmp/submission.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")
display(submission_df.head())

### Upload submission file on platform

Upload the `submission.csv` file on the challenge platform to generate your score on the leaderboard.


## Conclusion

We have evolved from a basic benchmark to a robust modeling framework by integrating multi-modal satellite data and key hydrological drivers from Phase 2 (PET, PR, and TMAX). By implementing a comparative analysis between RandomForest and HistGradientBoosting, we have moved beyond simple prediction toward a focus on model generalization and stability.

This pipeline serves as a launchpad for further innovation. We encourage you to experiment with additional feature engineering—such as lag variables for precipitation—or explore advanced hyperparameter optimization. We look forward to the insights uncovered as we continue to refine our approach to water quality prediction. Best of luck with the 2026 EY AI & Data Challenge!
